# 📊 AI CV Analyzer – Exploratory Data Analysis

Ce notebook explore les datasets Kaggle avant l'entraînement des modèles.

**Datasets nécessaires :**
- `../data/resumes.csv` (Kaggle Resume Dataset)
- `../data/job_descriptions.csv` (Job Descriptions Dataset)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import re

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('Libraries loaded ✓')

## 1. Load Resume Dataset

In [ ]:
try:
    df_resumes = pd.read_csv('../data/resumes.csv')
    print(f'Shape: {df_resumes.shape}')
    print(f'Columns: {df_resumes.columns.tolist()}')
    df_resumes.head(3)
except FileNotFoundError:
    print('⚠️ resumes.csv not found. Download from Kaggle first.')

## 2. Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
counts = df_resumes['Category'].value_counts()
counts.plot(kind='bar', ax=ax, color=sns.color_palette('husl', len(counts)))
ax.set_title('Distribution des catégories de CV', fontsize=16, fontweight='bold')
ax.set_xlabel('Catégorie')
ax.set_ylabel('Nombre de CV')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/eda_class_distribution.png', dpi=150)
plt.show()

## 3. Text Length Analysis

In [ ]:
df_resumes['word_count'] = df_resumes['Resume'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_resumes['word_count'], bins=50, edgecolor='white', color='steelblue')
axes[0].set_title('Distribution de la longueur des CV (mots)')
axes[0].set_xlabel('Nombre de mots')
axes[0].axvline(300, color='red', linestyle='--', label='Min ATS (300)')
axes[0].axvline(800, color='orange', linestyle='--', label='Max ATS (800)')
axes[0].legend()

df_resumes.boxplot(column='word_count', by='Category', ax=axes[1])
axes[1].set_title('Longueur par catégorie')
plt.suptitle('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(df_resumes['word_count'].describe())

## 4. Top Keywords per Category

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def get_top_keywords(df, category, n=15):
    texts = df[df['Category'] == category]['Resume'].tolist()
    if not texts:
        return []
    vec = TfidfVectorizer(stop_words='english', max_features=500, ngram_range=(1,2))
    X = vec.fit_transform(texts)
    scores = X.mean(axis=0).A1
    terms  = vec.get_feature_names_out()
    top_idx = scores.argsort()[::-1][:n]
    return [(terms[i], scores[i]) for i in top_idx]

# Pick 4 categories to display
sample_cats = df_resumes['Category'].value_counts().head(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for ax, cat in zip(axes, sample_cats):
    kws = get_top_keywords(df_resumes, cat)
    if kws:
        terms, scores = zip(*kws)
        ax.barh(terms[::-1], scores[::-1], color='steelblue')
        ax.set_title(f'Top keywords – {cat}', fontweight='bold')
        ax.set_xlabel('TF-IDF score')

plt.tight_layout()
plt.savefig('../data/eda_top_keywords.png', dpi=150)
plt.show()

## 5. Word Cloud – Full Dataset

In [ ]:
all_text = ' '.join(df_resumes['Resume'].dropna().tolist())
wc = WordCloud(width=1200, height=600, background_color='white',
               max_words=200, colormap='plasma').generate(all_text)

plt.figure(figsize=(16, 8))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud – CV Dataset', fontsize=20, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/eda_wordcloud.png', dpi=150)
plt.show()

## 6. Train / Test Split Check

In [ ]:
from sklearn.model_selection import train_test_split

X = df_resumes['Resume']
y = df_resumes['Category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)} ({len(X_train)/len(X):.0%})')
print(f'Test size:  {len(X_test)} ({len(X_test)/len(X):.0%})')
print(f'\nClass balance (train):\n{y_train.value_counts()}')

## 7. Quick Model Benchmark

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes':         MultinomialNB(),
    'Linear SVM':          LinearSVC(max_iter=2000),
}

results = {}
for name, clf in models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1,2))),
        ('clf',   clf),
    ])
    pipe.fit(X_train, y_train)
    acc = accuracy_score(y_test, pipe.predict(X_test))
    results[name] = acc
    print(f'{name:30s}: {acc:.4f} ({acc:.2%})')

# Plot
plt.figure(figsize=(8, 4))
plt.bar(results.keys(), results.values(), color=['#4C78A8', '#F58518', '#E45756'])
plt.ylim(0.8, 1.0)
plt.title('Comparaison des modèles de classification', fontweight='bold')
plt.ylabel('Accuracy')
plt.tight_layout()
plt.savefig('../data/eda_model_benchmark.png', dpi=150)
plt.show()